# Notebook 3: LSTM Deep Learning Model
Captures long-range temporal dependencies. Uses sliding window sequences.

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from evaluate import report
from visualize import plot_forecast

plt.rcParams['figure.figsize'] = (16, 5)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
df = pd.read_csv('../data/energy_clean.csv', index_col='timestamp', parse_dates=True)
series = df['consumption_mwh'].values.reshape(-1, 1)

scaler = MinMaxScaler()
scaled = scaler.fit_transform(series)
print(f'Total samples: {len(scaled):,}')

In [ ]:
SEQ_LEN = 168  # 1 week of hourly data as input
HORIZON  = 24  # predict next 24 hours

def make_sequences(data, seq_len, horizon):
    X, y = [], []
    for i in range(len(data) - seq_len - horizon + 1):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len:i+seq_len+horizon, 0])
    return np.array(X), np.array(y)

X, y = make_sequences(scaled, SEQ_LEN, HORIZON)
print(f'X shape: {X.shape}  |  y shape: {y.shape}')

# Train/val/test split (time-based)
n = len(X)
train_end = int(n * 0.80)
val_end   = int(n * 0.90)

X_train, y_train = X[:train_end], y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:], y[val_end:]
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

In [ ]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=5, monitor='val_loss')
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

model.save('../models/lstm_model.keras')

In [ ]:
# Training curves
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title('LSTM Training & Validation Loss')
plt.tight_layout()
plt.savefig('../outputs/figures/lstm_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate on test set (flatten multi-step predictions)
preds_scaled = model.predict(X_test)
preds_flat   = preds_scaled.flatten()
y_flat       = y_test.flatten()

# Inverse scale
preds_mwh = scaler.inverse_transform(preds_flat.reshape(-1,1)).flatten()
y_mwh     = scaler.inverse_transform(y_flat.reshape(-1,1)).flatten()

metrics = report(y_mwh, preds_mwh, 'LSTM')

# Plot first 7 days of test
n_plot = 168
plt.figure(figsize=(16, 5))
plt.plot(y_mwh[:n_plot], label='Actual', color='steelblue')
plt.plot(preds_mwh[:n_plot], label='Predicted', color='darkorange', linestyle='--')
plt.title('LSTM — First 7 Days of Test Set')
plt.ylabel('MWh')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/lstm_forecast.png', dpi=150, bbox_inches='tight')
plt.show()